# SMART BIO-AI — M1 Hb baseline

Research notebook for smartphone-image hemoglobin estimation. **No clinical claim is made.**

Pipeline: canonical metadata → explicit ROI → RGB/HSV/Lab features → patient-level split → XGBoost baseline → MAE/RMSE/R²/Pearson → Bland–Altman.

> Do not run this notebook until the real Hb dataset has been downloaded and mapped. Never invent target values or patient IDs.

In [ ]:
from pathlib import Path
import pandas as pd
from smartbio.hb_baseline import build_feature_table, train_test_hb, save_result

METADATA = Path('../data/canonical/hb.csv')
IMAGE_ROOT = Path('..')
OUT = Path('../reports/hb_baseline')

if not METADATA.exists():
    raise FileNotFoundError(f'Missing {METADATA}. Run canonicalization after mapping the real dataset.')

metadata = pd.read_csv(METADATA)
metadata.head()

## 1. Feature extraction

The existing SmartBio feature layer extracts robust statistics from RGB, HSV, Lab, chromaticity, ratios, intensity and saturation. ROI coordinates must be explicit for quantitative experiments.

In [ ]:
features = build_feature_table(metadata, image_root=IMAGE_ROOT, require_explicit_roi=True)
features.shape, features.head()

## 2. Patient-safe baseline

The train/test split is grouped by patient. This prevents multiple images from the same person leaking into both training and evaluation.

In [ ]:
result = train_test_hb(features, model_name='xgboost', random_state=42)
result.metrics

In [ ]:
save_result(result, OUT)
result.predictions.head()

## Interpretation

Report MAE, RMSE, R² and Pearson correlation together. Correlation alone is insufficient for agreement. Next milestones: compare Random Forest/SVR, add Bland–Altman, then CNN, quality gating, adaptive calibration, XAI and uncertainty.